In [ ]:
import os
import json
import torch
import pyNN.spiNNaker as sim
from abc import ABC, abstractmethod
from typing import List, Tuple

DATASET = "cifar10_dvs"
SIMULATION_TIME_MS = 20.0
DT = 1.0

WEIGHTS_PATHS = {
    "nmnist": "networks/nmnist_best.pth",
    "cifar10_dvs": "networks/cifar10_dvs_best.pth",
    "dvs_gesture": "networks/dvs_gesture_best.pth",
    "nepic_kitchens": "networks/nepic_kitchens_best.pth",
}

TOPOLOGIES = {
    "nmnist": [2312, 256, 10],
    "cifar10_dvs": [1568, 50176, 25088, 12544, 4608, 10],
    "dvs_gesture": [32768, 131072, 65536, 32768, 16384, 11],
    "nepic_kitchens": [116736, 262144, 131072, 65536, 32768, 16384, 8192, 4096, 8]
}

class ISpikeProvider(ABC):
    """
    Abstract interface for the activation time.
    """
    @abstractmethod
    def get_spikes(self) -> Tuple[List[List[float]], int]:
        pass

class JsonSpikeProvider(ISpikeProvider):
    """
    Implementation reading the event from a JSON file.
    """
    def __init__(self, file_path: str):
        self.file_path = file_path

    def get_spikes(self) -> Tuple[List[List[float]], int]:
        with open(self.file_path, 'r') as f:
            data = json.load(f)
        return data["spike_times"], data["label"]

def format_weights(weight_matrix: torch.Tensor) -> List[Tuple[int, int, float, float]]:
    """
    Transform a PyTorch weight matrice into connector pyNN compaibles.
    """
    connector_list = []
    if len(weight_matrix.shape) > 2:
        weight_matrix = weight_matrix.view(weight_matrix.size(0), -1)
        
    for i in range(weight_matrix.shape[1]):
        for j in range(weight_matrix.shape[0]):
            connector_list.append((i, j, float(weight_matrix[j, i]), 1.0))
    return connector_list

def deploy_real_spinnaker_inference(
    dataset_name: str, 
    pth_path: str, 
    spike_provider: ISpikeProvider, 
    simulation_time: float, 
    dt: float
) -> object:
    """Configure the network on SpiNNaker hardware and run the inference"""
    layers_sizes = TOPOLOGIES[dataset_name]
    
    state_dict = torch.load(pth_path, map_location="cpu", weights_only=True)
    if "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]
        
    spike_times, expected_label = spike_provider.get_spikes()
    
    sim.setup(timestep=dt)

    input_pop = sim.Population(
        layers_sizes[0], 
        sim.SpikeSourceArray(spike_times=spike_times)
    )
    
    populations = [input_pop]
    
    for size in layers_sizes[1:]:
        populations.append(sim.Population(size, sim.IF_curr_exp()))
        
    output_pop = populations[-1]
    output_pop.record(["spikes"])
    
    weight_keys = [k for k, v in state_dict.items() if len(v.shape) >= 2]
    
    for idx in range(len(populations) - 1):
        if idx < len(weight_keys):
            w_matrix = state_dict[weight_keys[idx]].cpu()
            connectors = format_weights(w_matrix)
            sim.Projection(
                populations[idx], 
                populations[idx+1], 
                sim.FromListConnector(connectors)
            )
        else:
            sim.Projection(populations[idx], populations[idx+1], sim.OneToOneConnector(weight=0.5))

    sim.run(simulation_time)
    spikes = output_pop.get_data("spikes")
    sim.end()
    
    print(f"Expected class (Ground Truth) for {dataset_name} : {expected_label}")
    return spikes

if __name__ == "__main__":
    pth_file = WEIGHTS_PATHS.get(DATASET, "")
    json_file = f"networks/sample_{DATASET}.json"
    
    if not os.path.exists(pth_file):
        raise FileNotFoundError(f"Weight file not found: {pth_file}")
    if not os.path.exists(json_file):
        raise FileNotFoundError(f"JSON file not found: {json_file}")
    
    provider = JsonSpikeProvider(json_file)
    
    results = deploy_real_spinnaker_inference(
        DATASET, 
        pth_file, 
        provider, 
        SIMULATION_TIME_MS, 
        DT
    )
    
    print(results.segments[0].spiketrains)